# Domain 3: Claude Code Configuration & Workflows
**Exam**: Claude Certified Architect – Foundations (CCAR-F)
**Weighting**: 20% of exam
**Depth**: deep

## Domain Overview

This domain covers configuring CLAUDE.md hierarchies, building custom slash commands and
skills, applying path-specific rules, choosing plan mode vs. direct execution, iterative refinement
techniques, and integrating Claude Code into CI/CD pipelines.

**Why it matters**: this is a 20%-weight domain testing whether an architect can configure Claude
Code for a real team, not just use it individually.

**Key concepts to know**: the CLAUDE.md hierarchy (user/project/directory), `.claude/rules/` with
glob-scoped frontmatter, `.claude/commands/` vs `.claude/skills/`, plan mode triggers, and the `-p`
flag plus `--output-format json` for CI usage.

In [ ]:
# Setup: these notebooks demonstrate Claude Agent SDK / Claude API / MCP concepts using
# plain Python simulations of the relevant objects and control flow, so every cell is runnable
# without needing a live Claude Agent SDK installation or API key. Where call_claude() is used,
# it falls back to a simulated response if ANTHROPIC_API_KEY is not set.
import os

def call_claude(prompt, system=None, model="claude-sonnet-4-5", max_tokens=600):
    try:
        import anthropic
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("no ANTHROPIC_API_KEY set")
        client = anthropic.Anthropic(api_key=api_key)
        kwargs = {"model": model, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        return resp.content[0].text
    except Exception:
        return f"[SIMULATED RESPONSE - set ANTHROPIC_API_KEY to call the real API] Prompt: {prompt[:120]}..."

print("call_claude() ready. Concepts below use plain Python to simulate Agent SDK / MCP / Claude Code behavior.")

## Key Concepts

### Configure CLAUDE.md files with appropriate hierarchy, scoping, and modular organization

CLAUDE.md exists at three levels: user (`~/.claude/CLAUDE.md`, personal, not shared),
project (`.claude/CLAUDE.md` or root `CLAUDE.md`, shared via version control), and directory-level
(subdirectory-specific). The `@import` syntax keeps large configs modular by referencing external
files, and `.claude/rules/` is an alternative to one giant monolithic file.

In [ ]:
claude_md_hierarchy = {
    "~/.claude/CLAUDE.md": "user-level - personal only, NOT shared with teammates via version control",
    ".claude/CLAUDE.md (or root CLAUDE.md)": "project-level - shared, version-controlled, applies to all",
    "src/api/CLAUDE.md": "directory-level - applies only within that subdirectory",
}
for path, meaning in claude_md_hierarchy.items():
    print(f"{path:30s} -> {meaning}")

# Diagnosing a common issue: new teammate not receiving team instructions
print("\nLikely cause if a new hire doesn't see team conventions: instructions were placed at USER level, not PROJECT level.")

### Create and configure custom slash commands and skills

Project-scoped commands live in `.claude/commands/` (shared via version control); user-scoped
commands live in `~/.claude/commands/` (personal). Skills live in `.claude/skills/` with SKILL.md
frontmatter options: `context: fork` (isolates verbose output in a sub-agent context),
`allowed-tools` (restricts tool access during the skill), and `argument-hint` (prompts for required
parameters).

In [ ]:
skill_frontmatter_example = """---
name: codebase-audit
context: fork
allowed-tools: [Read, Grep, Glob]
argument-hint: "<directory-to-audit>"
---
"""
print(skill_frontmatter_example)
print("context: fork -> isolates this skill's verbose exploration output from the main conversation")
print("allowed-tools -> restricts to read-only tools, preventing destructive actions during an audit")

### Apply path-specific rules for conditional convention loading

`.claude/rules/` files use YAML frontmatter with a `paths` field containing glob patterns
(e.g., `paths: ["**/*.test.tsx"]`) so rules load only when editing matching files — regardless of
which directory they live in. This is preferable to directory-level CLAUDE.md files when
conventions must apply to files spread throughout the codebase (like test files).

In [ ]:
rule_file_example = """---
paths: ["**/*.test.tsx", "**/*.test.ts"]
---
# Testing conventions
Use React Testing Library. Mock network calls with msw. One assertion focus per test.
"""
print(rule_file_example)

def rule_applies(rule_glob_pattern: str, filepath: str) -> bool:
    import fnmatch
    return fnmatch.fnmatch(filepath, rule_glob_pattern)

print(rule_applies("**/*.test.tsx", "src/components/Button.test.tsx"))
print(rule_applies("**/*.test.tsx", "src/components/Button.tsx"))

### Determine when to use plan mode vs. direct execution

Plan mode suits complex tasks: large-scale changes, multiple valid approaches, architectural
decisions, multi-file modifications. Direct execution suits simple, well-scoped changes (e.g., a
single-file bug fix with a clear stack trace). The Explore subagent isolates verbose discovery output
during multi-phase plan-mode work to preserve main-conversation context.

In [ ]:
def choose_execution_mode(files_affected: int, architectural_decision_needed: bool) -> str:
    if architectural_decision_needed or files_affected > 10:
        return "plan mode - explore and design before committing to changes"
    return "direct execution - well-scoped, clear change"

print(choose_execution_mode(files_affected=1, architectural_decision_needed=False))
print(choose_execution_mode(files_affected=45, architectural_decision_needed=True))

### Apply iterative refinement techniques for progressive improvement

Concrete input/output examples communicate expected transformations more reliably than
prose alone. Test-driven iteration (write tests first, iterate against failures) and the "interview
pattern" (have Claude ask clarifying questions before implementing) both improve results in
unfamiliar domains. Address multiple *interacting* issues in one message; fix independent issues
sequentially.

In [ ]:
def choose_refinement_technique(ambiguous_transformation: bool, unfamiliar_domain: bool, issues_interact: bool) -> str:
    if ambiguous_transformation:
        return "provide 2-3 concrete input/output examples"
    if unfamiliar_domain:
        return "use the interview pattern - have Claude ask clarifying questions first"
    if issues_interact:
        return "address all interacting issues in a single detailed message"
    return "fix issues sequentially, one at a time"

print(choose_refinement_technique(True, False, False))
print(choose_refinement_technique(False, True, False))
print(choose_refinement_technique(False, False, True))

### Integrate Claude Code into CI/CD pipelines

The `-p` (or `--print`) flag runs Claude Code non-interactively so it doesn't hang waiting
for input. `--output-format json` with `--json-schema` produces machine-parseable output for posting
inline PR comments. CLAUDE.md provides CI-invoked Claude Code with project context (testing
standards, fixture conventions).

In [ ]:
ci_command = 'claude -p "Review this PR for security issues" --output-format json --json-schema review_schema.json'
print(ci_command)
print("""
-p / --print       -> non-interactive mode, required for CI/CD (prevents hangs waiting for input)
--output-format json -> machine-parseable output for automated PR comment posting
--json-schema       -> enforces a specific structured output shape
""")

## Mini Hands-On Project

**Covers**: CLAUDE.md hierarchy, skills, path-specific rules, plan mode, CI integration.

**Scenario**: Configure Claude Code for a team repo with React components, an API layer, and a
test suite spread throughout the codebase.

### Step 1: Set up the CLAUDE.md hierarchy

In [ ]:
project_claude_md = "# Project conventions\nUse TypeScript strict mode. Prefer functional React components."
print("Placed at: .claude/CLAUDE.md (project-level, shared via version control)")
print(project_claude_md)

### Step 2: Add a path-specific rule for tests spread across the repo

In [ ]:
test_rule = rule_file_example  # reuse from the concept cell above
print(test_rule)
print(rule_applies("**/*.test.tsx", "src/features/billing/Invoice.test.tsx"))

### Step 3: Decide plan mode vs. direct execution for a described change

In [ ]:
change_description = "Migrate the state management library across the whole app (60+ files)."
print(choose_execution_mode(files_affected=60, architectural_decision_needed=True))

### Step 4: Wire it into CI

In [ ]:
print('claude -p "Review this PR against .claude/CLAUDE.md conventions" --output-format json --json-schema pr_review_schema.json')

## Lab Exercises

### Exercise 1: Diagnose a CLAUDE.md hierarchy bug

**Objective**: Diagnose why one teammate isn't receiving shared conventions others see.

**Produce**: A root-cause diagnosis and the fix.

In [ ]:
teammate_report = "I don't see the API error-handling conventions everyone else follows."
# TODO: name the likely root cause (which file/level is misconfigured) and the fix
root_cause = "..."  # TODO
fix = "..."  # TODO
print(root_cause, fix)

# Expected output: Diagnosis: conventions likely live in another teammate's ~/.claude/CLAUDE.md (user-level) instead of project-level.

In [ ]:
# STRETCH GOAL: write the exact project-level CLAUDE.md content that should be added to fix this permanently.
fix_content = "..."  # TODO
print(fix_content)

# Expected output: Concrete CLAUDE.md content moved to the project level, ensuring all teammates receive it.

### Exercise 2: Build a skill with proper frontmatter

**Objective**: Write a SKILL.md frontmatter block for a 'dependency-audit' skill.

**Produce**: A frontmatter block with context, allowed-tools, and argument-hint set appropriately.

In [ ]:
# TODO: write frontmatter for a skill that audits dependencies for vulnerabilities,
# should run in an isolated context, only needs read-only tools, and needs a directory argument
dependency_audit_frontmatter = """---
name: dependency-audit
context: ...
allowed-tools: [...]
argument-hint: "..."
---
"""
print(dependency_audit_frontmatter)

# Expected output: context: fork, read-only allowed-tools (e.g. Read, Grep, Bash for read-only commands), and a directory argument-hint.

In [ ]:
# STRETCH GOAL: write the skill body (after the frontmatter) describing the audit steps.
skill_body = "..."  # TODO
print(skill_body)

# Expected output: A short, clear set of audit steps the skill should perform.

### Exercise 3: Path-specific rule for a new file type

**Objective**: Write a rule that applies only to Terraform files anywhere in the repo.

**Produce**: A rule file with a glob pattern and test cases proving it matches/doesn't match correctly.

In [ ]:
# TODO: write a rule file (YAML frontmatter + body) scoped to all .tf files anywhere in the repo
terraform_rule = """---
paths: [...]
---
...
"""
print(terraform_rule)
print(rule_applies("terraform/**/*.tf", "modules/network/main.tf"))
print(rule_applies("terraform/**/*.tf", "src/app.py"))

# Expected output: The rule correctly matches .tf files and correctly excludes non-.tf files.

In [ ]:
# STRETCH GOAL: extend the rule to also apply to .tfvars files, and add a test case for that.
# TODO

# Expected output: Updated glob pattern matches both .tf and .tfvars files.

### Exercise 4: Plan mode decision drill

**Objective**: Classify 5 tasks as plan mode or direct execution.

**Produce**: 5 classifications with brief reasoning.

In [ ]:
tasks = [
    ("Add a null check to one function", 1, False),
    ("Migrate from REST to GraphQL across the app", 50, True),
    ("Fix a typo in an error message", 1, False),
    ("Choose between 2 caching strategies and implement one", 15, True),
    ("Add a single new API endpoint following existing patterns", 3, False),
]
# TODO: call choose_execution_mode(files_affected, architectural_decision_needed) for each and print result
for desc, files, arch in tasks:
    pass

# Expected output: Correct mode chosen for each task per the stated criteria.

In [ ]:
# STRETCH GOAL: for the one plan-mode task with the fewest files affected, explain in one sentence
# why file count alone would have been a misleading signal.

# Expected output: A sentence explaining that architectural complexity, not just file count, drove the plan-mode decision.

### Exercise 5: CI review pipeline design

**Objective**: Design a full CI command plus explain how to avoid duplicate comments on re-runs.

**Produce**: A CI command and a strategy for handling re-reviews after new commits.

In [ ]:
# TODO: write the CI command using -p, --output-format json, and --json-schema
ci_review_command = "..."  # TODO

# TODO: describe the strategy for avoiding duplicate comments when the review re-runs after new commits
rerun_strategy = "..."  # TODO
print(ci_review_command)
print(rerun_strategy)

# Expected output: A correct CI command, plus a strategy involving including prior findings in context and reporting only new/unaddressed issues.

In [ ]:
# STRETCH GOAL: sketch the JSON schema (as a Python dict) the --json-schema flag would enforce for review findings.
review_schema = {}  # TODO: e.g. {"type": "object", "properties": {"findings": {"type": "array", ...}}}
print(review_schema)

# Expected output: A plausible JSON schema shape for structured review findings (file, line, severity, message).

## Self-Assessment: Claude Code Configuration & Workflows

Answer each question, then check the Answers cell below.


**Q1.** Which CLAUDE.md level should hold conventions that must apply to every teammate on a project?

**Q2.** What does the `context: fork` frontmatter option do for a skill?

**Q3.** Why are `.claude/rules/` with glob patterns preferred over directory-level CLAUDE.md files for test conventions?

**Q4.** What kind of task should trigger plan mode rather than direct execution?

**Q5.** What CLI flag is required to run Claude Code non-interactively in a CI/CD pipeline?

## Answers


**A1.** Project-level (.claude/CLAUDE.md or root CLAUDE.md), since it is shared via version control to every teammate.

**A2.** It runs the skill in an isolated sub-agent context so its (often verbose) output doesn't pollute the main conversation.

**A3.** Because test files are typically spread across many directories, and glob-based rules apply based on file pattern regardless of location.

**A4.** Complex tasks involving large-scale changes, multiple valid approaches, or architectural decisions across many files.

**A5.** -p (or --print), which prevents the tool from hanging while waiting for interactive input.

---
**Next**: [Domain 4: Prompt Engineering & Structured Output](./04-prompt-engineering-and-structured-output.ipynb)
**Back**: [README](./README.md)